# MungNgoc – worker dịch phụ đề (Qwen 14B, Colab free)

Máy nhà đẩy file tiếng Trung lên Google Drive; notebook này dịch rồi ghi kết quả lại Drive. Không cần tunnel hay mở cổng.
Contract thư mục: xem `drive_sync/README.md`. Việc cần làm:

1. *Runtime → Change runtime type → T4 GPU*.
2. Trên máy nhà chạy `python3 -m drive_sync push-code` (đẩy `qwen_translate/` và `glossary.txt` lên Drive).
3. Chạy lần lượt các ô bên dưới. Ô cuối chạy mãi, dịch mọi file mới trong `inbox/`; bấm Stop khi xong.

In [ ]:
# 1. Kiểm tra GPU (cần thấy Tesla T4 ~15GB)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Nối Google Drive (cho phép truy cập khi được hỏi)
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/MungNgoc")   # đổi nếu bạn đặt DRIVE_REMOTE sang thư mục khác
for sub in ("inbox", "outbox", "status", "code"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
assert (ROOT / "code" / "qwen_translate").is_dir(), "Chưa thấy code/qwen_translate trên Drive: chạy `python3 -m drive_sync push-code` ở máy nhà."
print("Drive OK:", ROOT)

In [ ]:
# 3. Cài Ollama và chạy server ở nền
import os, subprocess, time, requests
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="24h")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2); print("Ollama đã chạy."); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama không khởi động được, xem /content/ollama.log")

In [ ]:
# 4. Tải model (~9GB, vài phút)
MODEL = "qwen2.5:14b-instruct"
!ollama pull $MODEL

In [ ]:
# 5. Worker: dịch mọi file mới trong inbox/, ghi kết quả vào outbox/ (ghi .tmp rồi đổi tên), báo trạng thái vào status/
import json, os, shutil, sys, time, traceback
sys.path.insert(0, str(ROOT / "code"))
from qwen_translate.translate import check_server, load_glossary, source_stem, translate_srt

HOST = "http://127.0.0.1:11434"
check_server(HOST, MODEL)
glossary = load_glossary(ROOT / "code" / "glossary.txt")
print("Glossary:", "có" if glossary else "KHÔNG CÓ (kiểm tra code/glossary.txt)")
inbox, outbox, work = ROOT / "inbox", ROOT / "outbox", Path("/content/work")
work.mkdir(exist_ok=True)

def heartbeat(busy):
    (ROOT / "status" / "worker.json").write_text(json.dumps({"time": time.time(), "model": MODEL, "busy": busy}))

while True:
    heartbeat(None)
    todo = [p for p in sorted(inbox.glob("*.srt"), key=lambda p: (len(p.name), p.name))
            if not p.name.endswith(".vi.srt")
            and not (outbox / f"{source_stem(p)}.vi.srt").exists() and not (outbox / f"{source_stem(p)}.fail.txt").exists()]
    for p in todo:
        stem = source_stem(p)
        heartbeat(stem)
        print(f"[{time.strftime('%H:%M:%S')}] dịch {stem} ...", flush=True)
        try:
            vi_path, failed = translate_srt(p, HOST, MODEL, glossary, out_dir=work)
            tmp = outbox / f"{stem}.vi.srt.tmp"
            shutil.copyfile(vi_path, tmp)
            os.replace(tmp, outbox / f"{stem}.vi.srt")
            print(f"    xong ({len(failed)} câu không dịch được)", flush=True)
        except Exception as e:
            (outbox / f"{stem}.fail.txt").write_text(f"{type(e).__name__}: {e}", encoding="utf-8")
            print("    LỖI:", e, flush=True); traceback.print_exc()
    time.sleep(20)